# bridge_grid

> Phase 20 (`specs/multiagent/phase20/`, experiment C1): the grids. `ARMS20` (Stage 4's `squads` arm, the 64-agent `colonyGuards` arm and four coupled arms: `coupled`, `bridge1`, `bridge2`, `bridgeGuard`); `c1Grid`, the 38 new 64-agent boards (23 with independent liars on Stage 1's seeds, and `C1_CLEAN`'s 15 clean boards, P20-14); `c1PilotGrid` (seed 1890); `reusedCells20`, the 16 reference boards replayed from the cache (P20-6); `c1Estimate` and `c1Range` (calls and cost per arm); part b's `squadEvalSuite` (32 agents in squads of 4, 3B, P20-10) and `reusedPartB` (Phase 17's one-board rows, configuration checked); `bridgeCheckCells` (P20-11) and `projectHours20`. The plans, the cost rule and the pilot check are in `bridge_plan`.

In [ ]:
#| default_exp bridge_grid

In [ ]:
#| hide
from nbdev.showdoc import *

## Arms and cells

In [ ]:
#| export
import csv
from pathlib import Path

from explRL.bench_grid import _fmt
from explRL.clue_colony import checkOptions
from explRL.clue_grid import clueCell
from explRL.colony4 import ARMS4, stage4Grid
from explRL.colony64 import stage64Grid
from explRL.colony_grid import COLONY_SEEDS, PILOT_SEED, WORLD18, colonyCell
from explRL.coop_grid import evalCell
from explRL.fix_grid import cellCalls
from explRL.llm_runs import LLMConfig, cellRow
from explRL.rl17_grid import BIG_SEEDS
from explRL.rl19_suite import world19e

_S = ARMS4['squads']                                     # colony, guards 'quarter', squads 8 (Stage 4's arm, unchanged)
_C = dict(_S, couple=0.25)
ARMS20 = dict(squads=_S, colonyGuards=ARMS4['colonyGuards'], coupled=_C, bridge1=dict(_C, bridges=1),
              bridge2=dict(_C, bridges=2), bridgeGuard=dict(_C, bridges=1, bridgeGuard=True))
SEEDS8, SEEDS5 = list(COLONY_SEEDS), list(COLONY_SEEDS[:5])                 # 1800-1807, 1800-1804
C1_CELLS = dict(coupled=SEEDS5, bridge1=SEEDS8, bridge2=SEEDS5, bridgeGuard=SEEDS5)   # 23 boards at N 64, independent liars
C1_CLEAN = dict(squads=SEEDS5, coupled=SEEDS5, bridge1=SEEDS5)   # P20-14: 15 boards at N 64, no liars, the 16 guards stay
PILOT20 = ('coupled', 'bridge1', 'bridgeGuard')                    # seed 1890, N 64, with liars; no clean pilot board
REUSED20 = (('squads', 'colony4'), ('colonyGuards', 'colony64'))   # 16 boards, seeds 1800-1807, with liars ($0, P20-6)
N20 = 64
PARTB_MODELS = ('warm',) + tuple(f'r17mix_s{s}' for s in range(1, 9))   # final adapters, as published (P20-10)


def c1Cell(study, arm, seed, fault='independent', runner='llm'):
    "One C1 cell at N 64, level '<arm>|<fault>' (fault 'clean': nLiar 0, the 16 guards in the first slots; P20-14), checked."
    c = colonyCell(study, arm, N20, seed, fault, runner, arms=ARMS20)
    checkOptions(c['cfg']['world'], N20, c['cfg'].get('nLiar', 0))
    return c


def c1Grid():
    "Study colony20: C1_CELLS with independent liars (23) and C1_CLEAN (15): the 38 new boards."
    return [c1Cell('colony20', a, s, f) for cs, f in ((C1_CELLS, 'independent'), (C1_CLEAN, 'clean'))
            for a, ss in cs.items() for s in ss]


def c1PilotGrid():
    "Study colony20Pilot: PILOT20 at N 64 with independent liars, seed 1890 (not analysed; P20-8)."
    return [c1Cell('colony20Pilot', a, PILOT_SEED) for a in PILOT20]


def reusedCells20(seeds=COLONY_SEEDS):
    "The 16 reference cells (P20-6): Stage 4's squads|independent at N 64 and the 64-agent run's colonyGuards|independent."
    keep = {f'{a}|independent' for a, _ in REUSED20}
    return [c for c in stage4Grid(seeds) + stage64Grid(seeds) if c['N'] == N20 and c['level'] in keep]

## Calls and costs

In [ ]:
#| export
PER_CALL4, ALLOW20 = 0.588 / 1920, 1.25    # Stage 4's 64-agent squads boards ($0.588, 1,920 calls); allowance on coupled arms
arm20 = lambda c: c['level'].split('|')[0]


def c1Estimate(cells, perCall):
    """Calls (cellCalls: 1,920 per liar board, 2,560 per clean board) and cost per arm. perCall: dollars per call, one number
    or {arm: dollars}; an arm without its own (the pilot has no squads or bridge2 board) takes the highest given (conservative)."""
    pc = perCall if isinstance(perCall, dict) else {}
    top = max(pc.values()) if pc else float(perCall)
    by = {}
    for c in cells:
        a, n = arm20(c), cellCalls(c)
        d = by.setdefault(a, dict(boards=0, calls=0, cost=0.0))
        d['boards'], d['calls'], d['cost'] = d['boards'] + 1, d['calls'] + n, d['cost'] + n * float(pc.get(a, top))
    return dict(boards=len(cells), calls=sum(d['calls'] for d in by.values()), cost=sum(d['cost'] for d in by.values()), byArm=by)


def c1Range(per=PER_CALL4, allow=ALLOW20):
    "The README's cost range before the pilot: (low, high) for the main run, the pilot and both; high: x allow on coupled arms."
    hi = {a: per * (allow if ARMS20[a].get('couple') else 1.0) for a in ARMS20}
    r = {k: [c1Estimate(g(), p)['cost'] for p in (per, hi)] for k, g in (('main', c1Grid), ('pilot', c1PilotGrid))}
    r['total'] = [m + p for m, p in zip(r['main'], r['pilot'])]
    return {**{k: tuple(round(x, 2) for x in v) for k, v in r.items()}, 'boards': len(c1Grid()) + len(c1PilotGrid())}

## Part b and the 3B bridge check

In [ ]:
#| export
COLAB = 'results/multiagent/rl/colab'
PARTB_FILES = dict(warm='eval_p17s0/warm.csv', **{m: f'eval_p17/{m}.csv' for m in PARTB_MODELS[1:]})


def squadEvalSuite(models=PARTB_MODELS):
    """Part b (P20-10, FR-C6): per model, 32 agents in squads of 4 on Phase 17's plain board (evalCell, which fixes the world;
    squads set after), seeds 900-909, clean ('<model>|n32sq4') and with 8 liars, one per squad ('<model>|n32sq4liar')."""
    cells = []
    for m in models:
        for test, kw in (('n32sq4', {}), ('n32sq4liar', dict(nLiar=8))):
            for sd in BIG_SEEDS:
                c = evalCell(m, test, 'blackboard', 32, sd, **kw)
                c['cfg']['world']['squads'] = 4
                cells.append(c)
    return cells


def _strRow(c):
    "The configuration columns of a cell's row as writeCSV writes them."
    return {k: _fmt(v) for k, v in cellRow(c, LLMConfig.fromDict(c['cfg']), {}).items()}


def reusedPartB(root=COLAB, models=PARTB_MODELS, files=PARTB_FILES):
    """The one-board rows of part b ('|n32', '|n32liar') from Phase 17's evaluation CSVs (raw strings), after a check that each
    row's configuration columns equal the matching new squads-of-4 cell's apart from `squads` and the level (ValueError)."""
    new = {(c['level'].replace('sq4', ''), c['seed']): _strRow(c) for c in squadEvalSuite(models)}
    out = []
    for m in models:
        for r in csv.DictReader(open(Path(root) / files[m], newline='')):
            if r['level'] not in (f'{m}|n32', f'{m}|n32liar'):
                continue
            want = new[(r['level'], int(float(r['seed'])))]
            bad = {k: (r.get(k), v) for k, v in want.items() if k not in ('level', 'world.squads') and
                   (_num(r.get(k)) != _num(v))}
            if bad:
                raise ValueError(f"{r['level']} seed {r['seed']}: configuration differs from the squads-of-4 cell: {bad}")
            out.append(r)
    if len(out) != 20 * len(models):
        raise ValueError(f'expected {20 * len(models)} one-board rows, found {len(out)}')
    return out


def _num(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return v


def bridgeCheckCells(seeds=BIG_SEEDS[:5]):
    """P20-11 (FR-C7): the warm start in A1's world (16 agents, counting board, r 2, 4 guards, 4 liars, one bridge per pair),
    in squads of 8 and of 4, seeds 900-904, 40 turns, notes memory; levels 'warm|bridge8' and 'warm|bridge4'."""
    cells = []
    for S in (8, 4):
        w = world19e(S, guards=4, couple=0.25, bridges=1)
        checkOptions(w, 16, 4)
        cells += [clueCell('c1Check', f'warm|bridge{S}', 16, sd, 'local', channel='blackboard', model='local:warm',
                           memory='notes', world=dict(w), turns=40, nLiar=4) for sd in seeds]
    return cells


def projectHours20(turnsPerHour=40_000, loadHours=0.4, unitsPerHour=12.0):
    """A100 hours (README's derivation): honest agent turns / turnsPerHour (Phase 17's step 0, model loading included) for
    part b's new cells and, plus loadHours of model loading, the bridge check; oneBoard: re-running the one-board arm."""
    turns = lambda cs: sum((c['N'] - c['cfg'].get('nLiar', 0)) * c['cfg']['turns'] for c in cs)
    t = dict(partB=turns(squadEvalSuite()), check=turns(bridgeCheckCells()))
    h = dict(partB=t['partB'] / turnsPerHour, check=t['check'] / turnsPerHour + loadHours, oneBoard=t['partB'] / turnsPerHour)
    h['total'] = h['partB'] + h['check']
    return dict(turns=t, hours=h, units=h['total'] * unitsPerHour)

## Tests

In [ ]:
# test_c1_grid (AC-10, FR-C1, FR-C2, FR-C6, FR-C7, P20-6, P20-14), the grids' half: 38 new cells (23 with liars, 15 clean),
# 3 pilot cells, 16 reused cells; calls 1,920 per liar board and 2,560 per clean board; the clean cells (nLiar 0, 16 guards,
# bridges dealt, no clean colonyGuards, the clean squads cell equal to Stage 4's apart from the fault and nLiar); the README's
# costs; checkOptions' errors; part b's 180 new cells; bridgeCheckCells (10); projectHours20. The plans, pilotCheck20 and
# costRule20 are in bridge_plan.
import copy
from collections import Counter

from explRL.bridge_board import dealBridges
from explRL.coop_grid import coopHash

g, gp, gr = c1Grid(), c1PilotGrid(), reusedCells20()
lv = Counter(c['level'] for c in g)
assert len(g) == 38 and len(gp) == 3 and len(gr) == 16 and all(c['N'] == 64 for c in g + gp + gr)
assert lv == {'coupled|independent': 5, 'bridge1|independent': 8, 'bridge2|independent': 5, 'bridgeGuard|independent': 5,
              'squads|clean': 5, 'coupled|clean': 5, 'bridge1|clean': 5}
assert sorted(c['seed'] for c in g if c['level'] == 'bridge1|independent') == list(range(1800, 1808))
assert {c['seed'] for c in g if c['level'] != 'bridge1|independent'} == set(range(1800, 1805))
assert {c['level'] for c in gp} == {f'{a}|independent' for a in PILOT20} and {c['seed'] for c in gp} == {1890}
assert Counter(c['level'] for c in gr) == {'squads|independent': 8, 'colonyGuards|independent': 8}
assert {c['study'] for c in gr} == {'colony4', 'colony64'} and not {c['level'] for c in g} & {'colonyGuards|clean'}
assert {cellCalls(c) for c in g if c['level'].endswith('|independent')} == {1920} and {cellCalls(c) for c in gp} == {1920}
clean = [c for c in g if c['level'].endswith('|clean')]
assert len(clean) == 15 and {cellCalls(c) for c in clean} == {2560}                     # AC-10
for c in clean:
    w = c['cfg']['world']
    assert c['cfg'].get('nLiar', 0) == 0 and w['guards'] == 16 and w['squads'] == 8 and 'fault' not in w
    if w.get('bridges'):
        assert dealBridges(64, 8, 0, 16, w['bridges']) == [[16 + s] for s in range(8)]    # bridges dealt with 0 liars
liarW = {c['level'].split('|')[0]: c['cfg']['world'] for c in g if c['level'].endswith('|independent')}
assert all(c['cfg']['world'] == liarW[c['level'].split('|')[0]] for c in clean if c['level'].split('|')[0] in liarW)
s4 = next(c for c in stage4Grid([1800]) if c['level'] == 'squads|independent' and c['N'] == 64)
sc = next(c for c in clean if c['level'] == 'squads|clean' and c['seed'] == 1800)
nl = lambda c: {k: v for k, v in c['cfg'].items() if k != 'nLiar'}
assert nl(s4) == nl(sc) and s4['cfg']['nLiar'] == 16 and sc['cfg']['nLiar'] == 0
assert liarW['bridge1'] == dict(s4['cfg']['world'], couple=0.25, bridges=1) and liarW['bridgeGuard']['bridgeGuard'] is True
# the README's costs: main $25.3-30.6, pilot $1.8-2.2, total $27.0-32.8, 41 new boards
R = c1Range()
assert R == dict(main=(25.28, 30.63), pilot=(1.76, 2.21), total=(27.05, 32.83), boards=41), R   # README: 25.3-30.6, 1.8-2.2, 27.0-32.8
e = c1Estimate(g, PER_CALL4)
assert e['calls'] == 23 * 1920 + 15 * 2560 and e['byArm']['squads'] == dict(boards=5, calls=12800, cost=12800 * PER_CALL4)
assert c1Estimate(g, {'coupled': 1e-4, 'bridge1': 3e-4})['byArm']['bridge2']['cost'] == 5 * 1920 * 3e-4   # missing arm: highest
# checkOptions' errors (FR-B3)
w0 = dict(g[0]['cfg']['world'])
for bad, N, nL in ((dict(w0, squads=None), 64, 16), ({k: v for k, v in w0.items() if k != 'couple'} | dict(bridges=1), 64, 16),
                   (dict(w0, bridges=2, bridgeGuard=True), 64, 16), (dict(w0, bridges=1, bridgeGuard=True, guards=0), 64, 16),
                   (dict(w0, guards=10, bridges=2), 16, 4)):
    try:
        checkOptions(bad, N, nL)
        raise AssertionError(f'accepted {bad}')
    except ValueError:
        pass
try:
    c1Cell('t', 'nonesuch', 1800)
    raise AssertionError('accepted an unknown arm')
except KeyError:
    pass
# part b's 180 new cells and the bridge check's 10 (P20-10, P20-11)
pb = squadEvalSuite()
assert len(pb) == 180 and Counter(c['level'].split('|')[1] for c in pb) == {'n32sq4': 90, 'n32sq4liar': 90}
assert {c['cfg']['world']['squads'] for c in pb} == {4} and {c['N'] for c in pb} == {32}
assert {c['cfg'].get('nLiar', 0) for c in pb if c['level'].endswith('liar')} == {8} and {c['seed'] for c in pb} == set(range(900, 910))
assert {c['level'].split('|')[0] for c in pb} == set(PARTB_MODELS) and len(PARTB_MODELS) == 9
assert 'squads' not in evalCell('warm', 'n32', 'blackboard', 32, 900)['cfg']['world']               # evalCell's world untouched
bc = bridgeCheckCells()
assert len(bc) == 10 and Counter(c['level'] for c in bc) == {'warm|bridge8': 5, 'warm|bridge4': 5}
assert all(c['cfg']['world']['guards'] == 4 and c['cfg']['nLiar'] == 4 and c['cfg']['world']['bridges'] == 1 and
           c['cfg']['world']['couple'] == 0.25 and c['cfg']['world']['colony'] and c['N'] == 16 for c in bc)
assert {c['cfg']['world']['squads'] for c in bc} == {8, 4} and {c['seed'] for c in bc} == set(range(900, 905))
H = projectHours20()
assert H['turns'] == dict(partB=201_600, check=4_800) and abs(H['hours']['partB'] - 5.04) < 1e-9 and 5.4 < H['hours']['total'] < 5.7

In [ ]:
# reusedPartB (FR-C6, P20-10, T4's done-when): the 180 one-board rows (9 models x 20: '|n32' and '|n32liar', seeds 900-909)
# whose configuration equals the new squads-of-4 cells' apart from `squads`; a changed configuration is refused. Needs the
# unpacked Colab results (results/multiagent/rl/colab/, not in git): skipped without them.
import os, shutil, tempfile

root = next((p for p in ('../' + COLAB, COLAB) if Path(p, 'eval_p17', 'r17mix_s1.csv').exists()), None)
if root:
    rb = reusedPartB(root)
    assert len(rb) == 180 and Counter(r['level'].split('|')[1] for r in rb) == {'n32': 90, 'n32liar': 90}
    assert {r['level'].split('|')[0] for r in rb} == set(PARTB_MODELS) and {int(float(r['seed'])) for r in rb} == set(range(900, 910))
    with tempfile.TemporaryDirectory() as d:                    # one row with a different world: refused
        for m, f in PARTB_FILES.items():
            os.makedirs(Path(d, f).parent, exist_ok=True)
            shutil.copy(Path(root, f), Path(d, f))
        p = Path(d, PARTB_FILES['r17mix_s3'])
        rs = list(csv.DictReader(open(p, newline='')))
        next(r for r in rs if r['level'] == 'r17mix_s3|n32')['world.R'] = '20'
        with open(p, 'w', newline='') as fh:
            w = csv.DictWriter(fh, fieldnames=list(rs[0]), lineterminator='\n')
            w.writeheader()
            w.writerows(rs)
        try:
            reusedPartB(d)
            raise AssertionError('accepted a changed configuration')
        except ValueError as e:
            assert 'world.R' in str(e)
    print(f'reusedPartB: {len(rb)} one-board rows from {root}')
else:
    print('reusedPartB: Colab results not unpacked here, skipped')

In [ ]:
# test_c1_hashes (NFR-1, AC-1): Phase 5-19 plan and configuration hashes unchanged by Phase 20 (recorded at 9fe0e5d, before
# T1; the cell configurations of Stage 1, the 64-agent run, Stage 4, Phase 17's team-size grid and Phase 19's contestSuite)
import importlib

from explRL.clue_rl import RLConfig, configHash
from explRL.colony_grid import stage1Grid
from explRL.rl17_grid import RL17_PLAN, configHash17, rl17Config, teamSizeSuite
from explRL.rl19_suite import contestSuite

HASHES = dict(
    bench_grid=dict(PREREG='8a59293847ae'), llm_grid=dict(ANALYSIS_PLAN='83df7591d1de'), clue_grid=dict(CLUE_PLAN='050a825d7bae'),
    liar_grid=dict(TRACE_PLAN='f5c25637acd0'), fix_grid=dict(FIX_PLAN='79abfb531cdd', HB_PLAN='791a2df079a8'),
    scale_grid=dict(TRUST_PLAN='6d2818ae84c2', SCALE_PLAN='b886baefb2ef'), followup_grid=dict(FOLLOWUP_PLAN='b5bea39be001'),
    coop_grid=dict(TRACKA_PLAN='45258d8a34da', RL_PLAN='4d1ea0078a6e', RL13_PLAN='3e50a4002c7b'), rl16_grid=dict(RL16_PLAN='082b3b7c9a31'),
    rl17_grid=dict(PILOT17_PLAN='2d3565364f29', RL17_PLAN='8e28b5fdfb1f'), rl18_grid=dict(RL18_PLAN='c103584f5aac'),
    colony_grid=dict(COLONY_PLAN='58e842277a3e', COLONY2_PLAN='2cde5619b0b3'), colony3=dict(COLONY3_PLAN='0c59fa10f2b0'),
    colony32=dict(COLONY32_PLAN='f51f0298d34a'), colony64=dict(COLONY64_PLAN='9764cd43e34c'), colony4=dict(STAGE4_PLAN='a441381ce078'),
    colony4b=dict(STAGE4B_PLAN='4d5d0a8e4f5b'), rl19_grid=dict(SPEED19='1729d8126b73', PILOT19_PLAN='7a61a61abfcc', RL19_PLAN='53d9cf04dd65'))   # SPEED19 (unfrozen): measures text corrected 2026-11-10 (a bridge reads its own board and its pair's mirror lines)
for m, ps in HASHES.items():
    mod = importlib.import_module(f'explRL.{m}')
    for n, h in ps.items():
        assert coopHash(getattr(mod, n)) == h, (m, n, coopHash(getattr(mod, n)))
cfgs = dict(stage1Grid=(stage1Grid(), 'cc4532d0ff20'), stage64Grid=(stage64Grid(), '8b4326ea327d'), stage4Grid=(stage4Grid(), 'c8750cbee289'),
            teamSizeSuite=(teamSizeSuite(['warm']), 'aff25f35a5cf'),
            contestSuite=(contestSuite(['warm', 'a19contest_s1'], 4, world='squads'), 'd6c1b9559015'))
for n, (cs, h) in cfgs.items():
    assert coopHash([c['cfg'] for c in cs]) == h, (n, coopHash([c['cfg'] for c in cs]))
assert configHash(RLConfig('m', 'o')) == '087f167412ae' and configHash17(rl17Config('r17mix', 1, 'm', 'o', RL17_PLAN['recipe'])) == '2118edee0e48'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()